# 去掉极值，能否减轻异常读数的影响？

比较普通平均与裁剪平均：两种方法处理相同读数，误差越小，估计越接近参考温度。
本例使用合成数据，展示说明、代码、图表与解释如何组成连贯 Notebook。

本页包含两个样式示例步骤：先检查指定异常，再检查没有指定异常时的代价。它们不是用户研究的默认路线或采用记录；实际共创按已讨论的当前步骤交付。本例可用已有 Python 3、IPython 与 Matplotlib 环境顺序运行。

## 步骤一：指定异常会怎样改变两种估计的误差？

先让同一组读数经过两种方法，比较绝对误差。这样可以看见被测改变如何作用于输入，而不是只看到一个最终分数。

### 1. 准备同一批读数

只给最后一次读数增加指定偏差，单位为摄氏度。其他读数和参考温度保持相同。

In [ ]:
from random import Random
from statistics import mean
from io import BytesIO
from IPython.display import Markdown, Image, display
from matplotlib.figure import Figure

reference = 20.0
n_readings = 9
seed = 7
offset = 8.0
trim_count = 1  # 从最高和最低各去掉多少个读数

In [ ]:
rng = Random(seed)
clean = [round(reference + rng.uniform(-0.4, 0.4), 2) for _ in range(n_readings)]
changed = clean.copy()
changed[-1] += offset
print("原始读数（°C）：", clean)
print("加入指定偏差后（°C）：", changed)

### 2. 定义裁剪平均，查看实际保留的输入

先排序，去掉两端各 `trim_count` 个读数，再求平均。普通平均使用全部输入；两种方法都不读取参考温度来调整估计。

In [ ]:
def trimmed_mean(values, trim_count):
    if trim_count < 0 or 2 * trim_count >= len(values):
        raise ValueError("裁剪数量需非负，并至少保留一个读数")
    ordered = sorted(values)
    retained = ordered[trim_count:len(ordered) - trim_count]
    return mean(retained)

In [ ]:
ordered = sorted(changed)
retained = ordered[trim_count:len(ordered) - trim_count]
print("排序后的读数：", ordered)
print("裁剪后保留的读数：", retained)
print("普通平均值：", round(mean(changed), 3))
print("裁剪平均值：", round(trimmed_mean(changed, trim_count), 3))

### 3. 用同一个指标比较

绝对误差是估计值与参考温度相差的度数。下面先定义计算，再用当前读数调用。结果表保留精确数值。

In [ ]:
def compare_estimators(values, reference, trim_count):
    estimates = {"普通平均": mean(values),
                 "裁剪平均": trimmed_mean(values, trim_count)}
    return [{"method": name, "estimate": estimate,
             "absolute_error": abs(estimate - reference)}
            for name, estimate in estimates.items()]

In [ ]:
def show_results(rows):
    table = ["| 方法 | 估计值（°C） | 绝对误差（°C）↓ |",
             "|:---|---:|---:|"]
    for row in rows:
        table.append(f"| {row['method']} | {row['estimate']:.3f} | "
                     f"{row['absolute_error']:.3f} |")
    display(Markdown("\n".join(table)))

changed_results = compare_estimators(changed, reference, trim_count)
show_results(changed_results)

### 4. 看清误差差异

横条越短，估计越接近参考温度。图中 Mean 对应普通平均，Trimmed 对应裁剪平均；横轴为绝对误差，单位是摄氏度。图和表使用同一份计算结果。

In [ ]:
def plot_errors(rows, title):
    figure = Figure(figsize=(6.5, 2.6), layout="constrained")
    axis = figure.subplots()
    errors = [row["absolute_error"] for row in rows]
    bars = axis.barh(["Mean", "Trimmed"], errors, height=0.5,
                     color=["#718598", "#248075"])
    axis.invert_yaxis()
    axis.bar_label(bars, fmt="%.3f", padding=5)
    axis.set_xlim(0, max(errors + [0.01]) * 1.3)
    axis.set_xlabel("Absolute error (°C)")
    axis.set_title(title)
    axis.spines[["top", "right", "left"]].set_visible(False)
    axis.tick_params(axis="y", length=0)
    buffer = BytesIO()
    figure.savefig(buffer, format="png", dpi=130)
    display(Image(data=buffer.getvalue()))

In [ ]:
plot_errors(changed_results, "Readings with a specified offset")
error_mean, error_trimmed = [row["absolute_error"] for row in changed_results]
print(f"当前偏差：{offset:g} °C；普通平均误差：{error_mean:.3f} °C；"
      f"裁剪平均误差：{error_trimmed:.3f} °C。")
print(f"普通平均误差减去裁剪平均误差：{error_mean-error_trimmed:.3f} °C。")

误差来自两种估计各自减去同一个参考温度，再取绝对值。上方差值为正时，裁剪平均在本次输入上误差更小；为负时则是普通平均更小。

普通平均保留指定偏差，其均值变化为 `offset / n_readings`。裁剪平均移除两端读数，可能去掉高端异常，同时也丢掉一个低端读数。排序与保留列表显示了这次实际移除了什么。

这一步检查了单次指定异常下的机制。要理解裁剪的代价，还需检查没有指定异常时会发生什么；下一步保持原始批次不变，只使用未添加偏差的读数。

## 步骤二：没有指定异常时，裁剪会付出什么代价？

上一步比较了加入偏差后的误差。现在使用同一批原始读数，保留方法、参考温度和指标，检查优势是否依赖异常存在。

### 1. 对原始读数计算相同指标

直接复用上一步的定义。没有重新采样，也没有根据本步结果修改方法。

In [ ]:
clean_results = compare_estimators(clean, reference, trim_count)
show_results(clean_results)

### 2. 查看原始条件下的误差

图例、单位和颜色沿用上一步。横轴范围按本组误差自适应；跨两图比较时读取标注数值，不比较条形像素长度。

In [ ]:
plot_errors(clean_results, "Original readings")
clean_mean, clean_trimmed = [row["absolute_error"] for row in clean_results]
print(f"原始读数：普通平均误差 {clean_mean:.3f} °C，"
      f"裁剪平均误差 {clean_trimmed:.3f} °C。")
print(f"普通平均误差减去裁剪平均误差：{clean_mean-clean_trimmed:.3f} °C。")

这里的误差仍由当前估计与参考温度直接计算。没有指定异常时，两端读数也可能是正常波动，裁剪会改变保留样本，因此误差不保证减少。

两步共同说明如何检查方法在有利条件与相应对照中的表现。本例只有一批合成数据，不能据此断言所有异常比例、波动水平或真实传感器任务都适用。若实际研究需要比较多个条件，应先讨论哪项变化会影响判断，再补充对应步骤。

主 Notebook 保留这两步的关键代码、图表和解释；在实际共创中，每步交付后结合用户对输入、比较和结果含义的反馈原位完善。